In [108]:
import anndata as ad
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from scipy import stats
from datetime import datetime

In [ ]:
# "indir" is a custom input path, and "outdir" is a custom output path.
# indir=""
# outdir=""

In [2]:
levels = ["01_threeclass_1vsOthers", "02_subclasses_in_neuron_1vsOthers", "03_subclasses_in_NN_1vsOthers"]

In [5]:
def get_segment_length(segment_):
    chrom_, start_, end_ = segment_.split("_")
    return int(end_) - int(start_)

In [ ]:
for level_ in tqdm(levels):
    for modification_ in tqdm(["5mC", "5hmC"]):
        original_diff_df=ad.read_h5ad(f'{indir}/{level_}/{modification_}G_{level_.split("_")[-2]}_frac_segment_diff.h5ad').to_df()
        # perserve only segements with lengths that are at least 200bp
        data_columns_1=pd.Series(original_diff_df.columns)
        data_columns_2=data_columns_1[data_columns_1.apply(get_segment_length)>=200]
        new_diff_df=original_diff_df[data_columns_2]
        ad.AnnData(new_diff_df).write_h5ad(f'{outdir}/{level_}/{modification_}G_{level_.split("_")[-2]}_frac_segment_diff.h5ad')

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

In [46]:
allc_name=lambda cellID : "allc_" + cellID

In [104]:
def adjust_p_value_withNaN(p_values):
    """for pd.Series p_values input"""
    # Step 1: Identify non-NaN indices
    valid_indices = ~np.isnan(p_values)
    valid_p_values = p_values[valid_indices]
    # Step 2: Perform BH correction on non-NaN values
    corrected_p_values=stats.false_discovery_control(valid_p_values, method="bh")
    # Step 3: Reinsert corrected values into the original array
    adjusted_p_values = np.full_like(p_values, np.nan)  # Start with an array of NaN
    adjusted_p_values[valid_indices] = corrected_p_values
    return(pd.Series(adjusted_p_values, index=p_values.index))

In [ ]:
for modification_ in tqdm(["5mC", "5hmC"]):
    print("****"+modification_+"****")
    print(f'{datetime.now()}\t Started reading {modification_}G_frac_segment_by_cell.h5ad ..')
    data=ad.read_h5ad(f"{indir}/{modification_}G_frac_segment_by_cell.h5ad").to_df()
    print(f'{datetime.now()}\t Finished reading {modification_}G_frac_segment_by_cell.h5ad ..')
    data_columns_1=pd.Series(data.columns)
    data_columns_2=data_columns_1[data_columns_1.apply(get_segment_length)>=200]
    data=data[data_columns_2]
    data_count=data.count(axis=0)
    print(f'{datetime.now()}\t Calculation of data_count finished ..\n')
    for level_ in tqdm(levels):
        print("-----"+level_+"-----")
        print(f'{datetime.now()}\t Started concating p-values of chromosomes')
        original_pvalue_df = pd.concat([ad.read_h5ad(f"{indir}/{level_}/result/{modification_}G_RankSumTest_result_chr{chr_number}.h5ad").to_df() for chr_number in range(1,20,1)], axis=1)
        original_pvalue_df2= original_pvalue_df.loc[original_pvalue_df.index.str.contains("pvalue"),:]
        print(f'{datetime.now()}\t Finished concating p-values of chromosomes')
        if level_=="01_threeclass_1vsOthers":
            label_="three_class_label"
            subclass_list=["Exc", "Inh", "Non"]
        else:
            label_="subclass_label"
            with open("../../../04.data/04.config_files/subclass_order_for_integration_with_zeng.txt", "rt") as f:
                subclass_list=f.read().split("\n")[:-1]
            if level_=="02_subclasses_in_neuron_1vsOthers":
                subclass_list=[i for i in subclass_list if "NN" not in i]
            elif level_=="03_subclasses_in_NN_1vsOthers":
                subclass_list=[i for i in subclass_list if "NN" in i]
        print(f'{datetime.now()}\t Started calculating mask df to mask cases where CellNumber<10 ..')
        meta_data=pd.read_csv("../../../04.data/02.metainfo/03.total/01.Young_Mouse/RNA_DNA_match_name_QC_class_label_young.csv",header=0)
        meta_data2=meta_data[meta_data["total_QC"]==1]
        meta_data3=meta_data2[[label_, modification_[1:]+'_ID']]
        meta_data_group=meta_data3.groupby(label_)
        data_dict_count={subclass_ : data.loc[allc_name(meta_data_group.get_group(subclass_)[modification_[1:]+'_ID'])].count(axis=0) \
                   for subclass_ in subclass_list}
        data_count_df1=pd.DataFrame(data_dict_count).T
        data_count_df2=data_count_df1.loc[subclass_list,:]
        data_count_df3=(data_count_df2>=10)*((data_count-data_count_df2)>=10)
        print(f'{datetime.now()}\t Finished calculating mask df to mask cases where CellNumber<10 ..')
        mask_df = data_count_df3.map(lambda x: np.float32(np.nan) if not x else np.float32(x))
        # data_count_df3 is of dtype bool, mannually asign np.float32 to avoid mask_df acquiring dtypes of "object"
        mask_df.index=original_pvalue_df2.index

        masked_pvalue_df = original_pvalue_df2 * mask_df
        print(f'{datetime.now()}\t Finished masking ..')
        adjusted_masked_pvalue_df = masked_pvalue_df.apply(adjust_p_value_withNaN, axis=1)
        print(f'{datetime.now()}\t Finished BH adjusting ..')
        ad.AnnData(adjusted_masked_pvalue_df).write_h5ad(f'{outdir}/{level_}/{modification_}G_{level_.split("_")[-2]}_RankSumTest1vsOthers_AdjustedMasked_pvalue.h5ad')
        print(f'{datetime.now()}\t Saved {modification_}G_{level_.split("_")[-2]}_RankSumTest1vsOthers_AdjustedMasked_pvalue.h5ad')

  0%|          | 0/2 [00:00<?, ?it/s]

****5mC****
2024-12-15 17:31:58.909298	 Started reading 5mCG_frac_segment_by_cell.h5ad ..
2024-12-15 17:37:44.536510	 Finished reading 5mCG_frac_segment_by_cell.h5ad ..
2024-12-15 17:43:25.850359	 Calculation of data_count finished ..



  0%|          | 0/3 [00:00<?, ?it/s]

-----01_threeclass_1vsOthers-----
2024-12-15 17:43:25.858743	 Started concating p-values of chromosomes
2024-12-15 17:43:44.045544	 Finished concating p-values of chromosomes
2024-12-15 17:43:44.045625	 Started calculating mask df to mask cases where CellNumber<10 ..
2024-12-15 17:50:44.243639	 Finished calculating mask df to mask cases where CellNumber<10 ..
2024-12-15 17:51:21.152052	 Finished masking ..
2024-12-15 17:51:22.332432	 Finished BH adjusting ..
2024-12-15 17:51:23.058277	 Saved 5mCG_threeclass_RankSumTest1vsOthers_AdjustedMasked_pvalue.h5ad
-----02_subclasses_in_neuron_1vsOthers-----
2024-12-15 17:51:23.059047	 Started concating p-values of chromosomes
2024-12-15 17:51:25.582526	 Finished concating p-values of chromosomes
2024-12-15 17:51:25.582836	 Started calculating mask df to mask cases where CellNumber<10 ..
2024-12-15 17:56:28.469644	 Finished calculating mask df to mask cases where CellNumber<10 ..
2024-12-15 17:57:20.064525	 Finished masking ..
2024-12-15 17:57:28

  0%|          | 0/3 [00:00<?, ?it/s]

-----01_threeclass_1vsOthers-----
2024-12-15 18:10:03.656231	 Started concating p-values of chromosomes
2024-12-15 18:10:19.764060	 Finished concating p-values of chromosomes
2024-12-15 18:10:19.764143	 Started calculating mask df to mask cases where CellNumber<10 ..
2024-12-15 18:12:45.695671	 Finished calculating mask df to mask cases where CellNumber<10 ..
2024-12-15 18:13:08.114083	 Finished masking ..
2024-12-15 18:13:08.769742	 Finished BH adjusting ..
2024-12-15 18:13:09.217380	 Saved 5hmCG_threeclass_RankSumTest1vsOthers_AdjustedMasked_pvalue.h5ad
-----02_subclasses_in_neuron_1vsOthers-----
2024-12-15 18:13:09.218149	 Started concating p-values of chromosomes
2024-12-15 18:13:10.756054	 Finished concating p-values of chromosomes
2024-12-15 18:13:10.756330	 Started calculating mask df to mask cases where CellNumber<10 ..
2024-12-15 18:15:27.235583	 Finished calculating mask df to mask cases where CellNumber<10 ..
2024-12-15 18:15:59.662106	 Finished masking ..
2024-12-15 18:16:0

In [203]:
np.sum(np.sum(masked_pvalue_df.isnull()))

/share/home/renlh/miniconda3/envs/default/lib/python3.12/site-packages/numpy/core/fromnumeric.py:86: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


61450

In [204]:
np.sum(np.sum(original_pvalue_df2.isnull()))

26621

In [205]:
np.sum(np.sum(mask_df.isnull()))

61450

In [206]:
np.sum(np.sum(original_pvalue_df2.isnull())<=np.sum(mask_df.isnull()))

831303

In [ ]:
for modification_ in tqdm(["5mC", "5hmC"]):
    print("****"+modification_+"****")
    if modification_ == "5mC":
        diff_threshold_ = 0.3
    elif modification_ == "5hmC":
        diff_threshold_ = 0.2
    else:
        raise Exception("wrong modification")
        
    for level_ in tqdm(levels):
        print("-----"+level_+"-----")
        df_pvalue=ad.read_h5ad(f'{indir}/{level_}/{modification_}G_{level_.split("_")[-2]}_RankSumTest1vsOthers_AdjustedMasked_pvalue.h5ad').to_df()
        df_diff=ad.read_h5ad(f'{indir}/{level_}/{modification_}G_{level_.split("_")[-2]}_frac_segment_diff.h5ad').to_df()
        the_index=pd.Series(df_pvalue.index).apply(lambda x : x.replace("_Others_pvalue", ""))
        df_pvalue.index=the_index
        df_diff.index=the_index
        for pvalue_threshold_ in [0.05]:
            df_DMR_states=np.sign(df_diff) * (df_diff.abs()>diff_threshold_) * (df_pvalue <= pvalue_threshold_)
            print(modification_, level_, pvalue_threshold_)
            print(np.sum(df_DMR_states.abs().sum(axis=0)>0))
            ad.AnnData(df_DMR_states).write_h5ad(f'{outdir}/{level_}/{modification_}G_DMR_states_{level_.split("_")[-2]}_1vsOthers.h5ad')
        

  0%|          | 0/2 [00:00<?, ?it/s]

****5mC****


  0%|          | 0/3 [00:00<?, ?it/s]

-----01_threeclass_1vsOthers-----
5mC 01_threeclass_1vsOthers 0.05
17773
-----02_subclasses_in_neuron_1vsOthers-----
5mC 02_subclasses_in_neuron_1vsOthers 0.05
133659
-----03_subclasses_in_NN_1vsOthers-----
5mC 03_subclasses_in_NN_1vsOthers 0.05
54985
****5hmC****


  0%|          | 0/3 [00:00<?, ?it/s]

-----01_threeclass_1vsOthers-----
5hmC 01_threeclass_1vsOthers 0.05
39557
-----02_subclasses_in_neuron_1vsOthers-----
5hmC 02_subclasses_in_neuron_1vsOthers 0.05
240150
-----03_subclasses_in_NN_1vsOthers-----
5hmC 03_subclasses_in_NN_1vsOthers 0.05
9567
